# Falnama — Analyst Review

A compiled, visual view of a pipeline run, so you don't have to read raw files in
an editor. All parsing lives in `falnama/review.py`; this notebook just displays.

**Setup:** `pip install -e ".[viz]"` (adds jupyter + matplotlib), then run all cells.
By default it loads the **latest** run — pass a `run_id` to `review.load_run(...)`
to inspect any past run.

In [ ]:
import sys; sys.path.insert(0, "..")   # import falnama when running from notebooks/
import pandas as pd
import matplotlib.pyplot as plt
from falnama import review

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

run = review.load_run()                # latest run; or review.load_run(run_id="20260715T092325Z")
m = run.manifest
print(f"Run {run.run_id}  |  source={m.get('data_source')}  card_mode={m.get('card_mode')}  |  success={run.health.get('success')}")

## 1. Run summary

In [ ]:
sel, an, rec = m["market_selector"], m["anomaly_detector"], m["recommender"]
print(f"selected markets   : {sel['selected_count']:>4}   (rejected {sel['rejected_count']})")
print(f"anomalies scored   : {an['markets_scored']:>4}   (strong {an['strong_count']}, concentration red-flags {an['concentration_red_flags']})")
print(f"paper recommends   : {rec['recommended_count']:>4}   (rejected signals {rec['rejected_count']})")
if run.health.get("warnings"): print("warnings:", run.health["warnings"])
if run.health.get("errors"):   print("errors  :", run.health["errors"])

## 2. Selected market universe
What passed Stage 1, and how it breaks down by topic and region.

In [ ]:
rel = run.relevant_markets
cols = [c for c in ["market_name","primary_topic","country_or_region","relevance_score"] if c in rel.columns]
display(rel[cols].sort_values("relevance_score", ascending=False).head(20))

if not rel.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    rel["primary_topic"].value_counts().plot.barh(ax=ax[0], title="markets by topic")
    rel["country_or_region"].value_counts().plot.barh(ax=ax[1], title="markets by region")
    ax[0].invert_yaxis(); ax[1].invert_yaxis(); plt.tight_layout(); plt.show()

## 3. Anomalies
Ranked by the composite score. `strong` is the class the recommender may act on.

In [ ]:
an_df = run.anomalies
cols = [c for c in ["market_name","anomaly_score","anomaly_class","concentration_tier","max_abs_move"] if c in an_df.columns]
display(an_df[cols].sort_values("anomaly_score", ascending=False).head(15))

if not an_df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    order = ["strong", "medium", "weak", "subthreshold"]
    an_df["anomaly_class"].value_counts().reindex(order).fillna(0).plot.bar(ax=ax[0], title="anomalies by class", color="#c0504d")
    an_df["concentration_tier"].value_counts().plot.bar(ax=ax[1], title="concentration tiers", color="#4f81bd")
    for a in ax: a.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()

## 4. Paper recommendations & rejected signals
Zero recommendations is a normal, valid outcome (the no-trade bias).

In [ ]:
print("Recommendations:")
display(run.recommendations if not run.recommendations.empty else "— none this run —")
print("\nRejected signals (why candidates were dropped):")
rej = run.rejected_signals
cols = [c for c in ["market_name","anomaly_score","reason"] if c in rej.columns]
display(rej[cols] if not rej.empty else "— none this run —")

## 5. Cross-run trends
How the pipeline's numbers move over time — useful for spotting drift or the effect of a config change.

In [ ]:
hist = review.run_history()
display(hist)

if len(hist) > 1:
    h = hist.copy(); h["day"] = h["run_id"].str[:8]
    ax = h.plot(x="day", y=["selected", "scored", "strong", "recommended"], marker="o",
                figsize=(11, 4), title="pipeline metrics per run")
    ax.set_xlabel("run"); plt.tight_layout(); plt.show()